### Explanation of Embeddings


In [ ]:
import numpy as np 
# NumPy is Python’s most popular library for numerical computing.
#It provides a powerful object called an ndarray (N-dimensional array)
#We use numpy to store embeddings as number as numpy are faster 

from sentence_transformers import SentenceTransformer
#It is used to convert text into embeddings using hugging face model - "all-MiniLM-L6-v2"

import chromadb
from chromadb.config import Settings
#Use to configure chroma db the vectorDb 

import uuid
#Universally unique identifier - every emebedding is given a unique identity 

from typing import List, Dict, Any, Tuple
#These don’t affect execution.
#They help humans and editors understand what a function expects and returns.

from sklearn.metrics.pairwise import cosine_similarity
#Use to find similarity between diffrent data like i ask a Q - "why is burj khalifa so tall" && "Why was burj khalifa made so tall" so these are similar that will be found by cosine similarity 

In [ ]:
class EmbeddingHead:
    """Handles document embedding generation using SentenceTransformer"""  # Class responsible for loading an embedding model and generating text embeddings.

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initialize the embedding head.

        Args:
            model_name: HuggingFace model name for sentence embeddings.
        """

        self.model_name = model_name      # Store the embedding model name inside the object.
        self.model = None                 # Initially, no model is loaded into memory.
        self._load_model()                # Automatically load the model when an object is created.

    def _load_model(self):
        """Load the SentenceTransformer model."""

        try:
            print(f"Loading embedding model: {self.model_name}")  # Display which model is being loaded.

            self.model = SentenceTransformer(self.model_name)     # Download/load the embedding model from Hugging Face.

            print(
                f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}"
            )                                                    # Print the embedding vector size (e.g., 384).

        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")  # Display the actual error message.
            raise                                                 # Re-raise the exception to stop execution.

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:    #Convert text into embeddings return type is numpy array
        
        """Generate embeddings for a list of texts.

        Args:
            texts: List of text strings to embed.

        Returns:
            NumPy array of embeddings with shape
            (number_of_texts, embedding_dimension).
        """

        if not self.model:
            raise ValueError("Model not loaded")                  # Prevent embedding generation if the model failed to load.

        print(f"Generating embeddings for {len(texts)} texts...") # Display the number of texts being processed.

        embeddings = self.model.encode(                           # Encode is the fnc used on texts 
            texts,
            show_progress_bar=True
        )                                                         # Convert each text into a numerical embedding vector.

        print(f"Generated embeddings with shape: {embeddings.shape}")  # Example: (25, 384)

        return embeddings                                         # Return the generated embedding vectors.


# -------------------- Example Usage --------------------

embeddings = EmbeddingHead()      # Create an EmbeddingHead object. This automatically loads the embedding model.

embeddings                        # Display the created object in Jupyter Notebook.

## Explanation of Vector store

In [ ]:
class VectorStore:

    """Manages document embeddings in a ChromaDB vector store"""
    # This class handles all vector database operations such as
    # creating the database, storing embeddings, and managing documents.

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):

        """Initialize the Vector Store.

        Args:
            collection_name: Name of the ChromaDB collection.
            persist_directory: Directory where the vector database will be stored.
        """

        self.collection_name = collection_name
        # Store the collection name so it can be accessed by all methods of this class.

        self.persist_directory = persist_directory
        # Store the directory path where ChromaDB will persist (save). embeddings and metadata on the local machine.

        self.client = None
        # Placeholder for the ChromaDB client. It will be initialized, once a connection to the vector database is established.

        self.collection = None
        # Placeholder for the ChromaDB collection that will store document embeddings and their metadata.

        self._initialize_store()
        # Automatically initialize the vector store when the object is created, making it ready to use immediately.

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""

        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            # Create the persistence directory if it doesn't already exist.
            # exist_ok=True prevents an error if the folder is already present.

            self.client = chromadb.PersistentClient(path=self.persist_directory)
            # Create a persistent ChromaDB client that stores the vector
            # database on disk instead of only in memory.

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            # Retrieve the collection if it already exists;
            # otherwise create a new one with the given name.

            print(f"Vector store initialized. Collection: {self.collection_name}")
            # Display the name of the initialized collection.

            print(f"Existing documents in collection: {self.collection.count()}")
            # Show the number of documents already stored in the collection.

        except Exception as e:
            
            print(f"Error initializing vector store: {e}")
            # Print the error message if vector store initialization fails.

            raise
            # Re-raise the exception so the program stops instead
            # of continuing with an invalid vector store.

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):

        """
        Add documents and their embeddings to the vector store.

        Args:
            documents: List of LangChain Document objects.
            embeddings: Corresponding embedding vectors for each document.
        """

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
            # Ensure that every document has exactly one corresponding embedding.

        print(f"Adding {len(documents)} documents to vector store...")
        # Display how many documents are about to be stored.

        # Prepare lists required by ChromaDB.
        ids = []                # Stores unique IDs for every document.
        metadatas = []          # Stores metadata associated with each document.
        documents_text = []     # Stores the actual document text.
        embeddings_list = []    # Stores embeddings as Python lists.

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Iterate through documents and embeddings together while
            # keeping track of each document's index.

            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            # Generate a unique ID using UUID and append the index
            # to avoid duplicate document identifiers.

            ids.append(doc_id)
            # Save the generated ID.

            metadata = dict(doc.metadata)
            # Create a copy of the document's metadata so it can
            # be modified without changing the original object.

            metadata["doc_index"] = i
            # Store the document index for easy identification.

            metadata["content_length"] = len(doc.page_content)
            # Store the length of the document text as additional metadata.

            metadatas.append(metadata)
            # Save the metadata for this document.

            documents_text.append(doc.page_content)
            # Store the actual text content of the document.

            embeddings_list.append(embedding.tolist())
            # Convert the NumPy embedding into a regular Python list,
            # as required by ChromaDB.

        try:

            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            # Add all documents, embeddings, metadata, and IDs
            # to the ChromaDB collection in a single operation.

            print(f"Successfully added {len(documents)} documents to vector store")
            # Confirm that all documents were stored successfully.

            print(f"Total documents in collection: {self.collection.count()}")
            # Display the updated number of documents in the collection.

        except Exception as e:

            print(f"Error adding documents to vector store: {e}")
            # Print the error message if adding documents fails.

            raise
            # Re-raise the exception so the calling code can handle it properly.


# ---------------- Example Usage ----------------

vectorstore = VectorStore()
# Create a VectorStore object. This automatically initializes
# the ChromaDB client and creates/loads the collection.

vectorstore
# Display the created VectorStore object in a Jupyter Notebook.

## Retrieval pipeline

In [ ]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    # This class is responsible for retrieving the most relevant document
    # chunks from the vector database based on the user's query.

    def __init__(self, vector_store: VectorStore, embeddings: EmbeddingHead):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embeddings: Manager for generating query embeddings
        """

        self.vector_store = vector_store
        # Store the VectorStore object so this class can access the
        # ChromaDB collection whenever a search query is made.

        self.embeddings = embeddings
        # Store the EmbeddingHead object which will be used to convert
        # user queries into embedding vectors before retrieval.

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """

        print(f"Retrieving documents for query: '{query}'")
        # Display the user's query so we know what is currently
        # being searched in the vector database.

        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        # Show the retrieval settings, including how many documents
        # should be returned and the minimum similarity score allowed.

        # Generate query embedding
        query_embedding = self.embeddings.generate_embeddings([query])[0]
        # Convert the user's query into a numerical embedding vector.
        # The method returns a list of embeddings, so [0] extracts the embedding for our single query.

        # Search in vector store
        try:

            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            # Search the ChromaDB collection using the query embedding.
            # The embedding is converted to a Python list because ChromaDB expects list inputs instead of NumPy arrays.

            # Process results
            retrieved_docs = []
            # Create an empty list that will store the retrieved
            # documents along with their metadata and similarity scores.

            if results['documents'] and results['documents'][0]:
                # Check whether ChromaDB actually returned any matching
                # documents before trying to process them.

                documents = results['documents'][0]
                # Extract the retrieved document texts from the results.
                # Since there is only one query, the results are stored inside the first list.

                metadatas = results['metadatas'][0]
                # Extract the metadata corresponding to each retrieved
                # document such as page number, source file, etc.

                distances = results['distances'][0]
                # Extract the cosine distances returned by ChromaDB.
                # Smaller distance means the retrieved document is more similar to the query.

                ids = results['ids'][0]
                # Extract the unique IDs assigned to each retrieved
                # document when they were inserted into the vector store.

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Iterate through all retrieved results together while
                    # keeping track of each document's rank using enumerate().

                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    # Convert cosine distance into a similarity score.
                    # A higher similarity score indicates a more relevant document.

                    if similarity_score >= score_threshold:
                        # Only keep documents whose similarity score is
                        # greater than or equal to the minimum threshold.

                        retrieved_docs.append({
                            'id': doc_id,
                            # Store the unique identifier of the retrieved document.

                            'content': document,
                            # Store the actual text content that matched the query.

                            'metadata': metadata,
                            # Store metadata like page number, source file,
                            # document index, etc., for future reference.

                            'similarity_score': similarity_score,
                            # Store the computed similarity score so we know
                            # how relevant this document is to the query.

                            'distance': distance,
                            # Store the original cosine distance returned by
                            # ChromaDB for debugging or further analysis.

                            'rank': i + 1
                            # Store the ranking position of the document.
                            # Rank starts from 1 instead of 0 for readability.
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
                # Display how many documents satisfied the similarity
                # threshold and will be returned to the user.

            else:
                print("No documents found")
                # Inform the user that no relevant documents were found
                # for the given query.

            return retrieved_docs
            # Return the final list of retrieved documents along with
            # their metadata and similarity information.

        except Exception as e:

            print(f"Error during retrieval: {e}")
            # Print the error message if anything goes wrong during
            # retrieval so it is easier to debug.

            return []
            # Return an empty list instead of crashing the program,
            # allowing the caller to handle the failure gracefully.


rag_retriever = RAGRetriever(vectorstore, embeddings)
# Create an object of the RAGRetriever class by passing the existing
# VectorStore and EmbeddingHead objects. The retriever can now generate
# query embeddings and search the vector database for relevant documents.